# Genotype alignment

**Aim.** Align genotype `bim` files from multiple cohorts to a single reference cohort (ROSMAP) so that variant alleles (`alt`/`ref`) are made consistent across studies, with allele flipping handled automatically.

**Method.** The `genotype_alignment` workflow walks each provided cohort folder, collects per-chromosome `bim` files, groups them by chromosome, treats the first cohort as the reference, and merges every subsequent cohort onto it (matching on chromosome+position with order-independent allele keys and flipping alleles where needed). The aligned table is written as a bgzip-compressed, tabix-indexed `bim`.

## Input

For processing different `bim files` from multiple cohorts, there should be a predefined order, with ROSMAP being the first. This implies that each subsequent cohort should align with ROSMAP first, and then with the following cohorts in the specified sequence.

For this minimal working example, two small toy cohort folders are provided under `input/`, each containing per-chromosome `bim` files (chr21, chr22) named `*.<chrom>.bim`:

- `input/genotype/protocol_example.geno_cohortA` (reference cohort, listed first)
- `input/genotype/protocol_example.geno_cohortB` (second cohort, aligned to the first)

## Output

1. Bgzip-compressed files, aligned across all cohort bim files, with columns are `chromosome`, `position`, `alt`, and `ref`, but without colnames. 
```
zcat demo.ROSMAP_NIA_WGS.MSBB_WGS_ADSP_hg38.11.aligned.bim.gz|head
11	70548	AG	A
11	70560	T	C
11	70564	T	C
11	70575	T	C
11	70576	A	G
11	70582	C	A
11	70582	G	A
```


2. `.tbi` index from tabix

## Steps

**Step 1.** Align the cohorts to the reference (first folder) and write bgzip-compressed, tabix-indexed aligned `bim` files. List the reference cohort folder first.

**Timing:** ~5-10 min on typical compute infrastructure.

In [ ]:
sos run pipeline/genotype_alignment.ipynb genotype_alignment \
    --geno_list_paths input/genotype/protocol_example.geno_cohortA input/genotype/protocol_example.geno_cohortB \
    --cwd output/genotype_alignment \
    --name protocol_example

## Command interface

In [ ]:
sos run pipeline/genotype_alignment.ipynb -h

## Workflow implementation

## Anticipated Results

The pipeline produces output files in the `output/` subdirectory named after the workflow step. Verify success by checking that output files exist and are non-empty. See the **Output** section above for the expected file names and formats.

In [ ]:
[global]
import glob
import pandas as pd
## Path to work directory where output locates
parameter: cwd = path("output")
parameter: name = str
## Containers that contains the necessary packages
parameter: container = ""
# For cluster jobs, number commands to run per job
parameter: job_size = 50
# Wall clock time expected
parameter: walltime = "96h"
# Memory expected
parameter: mem = "6G"
# Number of threads
parameter: numThreads = 2
parameter: windows = 1000000
# use this function to edit memory string for PLINK input
from sos.utils import expand_size

In [1]:
#align different cohort's genotype file to ROSMAP's. chr11 on ROSMAP and MIGA costed 4.5h
[genotype_alignment]
# Input
# A list of folder paths with bim files, with orders, rosmap should be first, and then maybe mssb. 
parameter: geno_list_paths = []
# a function to split the genofiles by chrom
import pandas as pd
import os
import re

def group_by_region(lst, partition):
    return partition

bim_files = []

for path in geno_list_paths:
    for root, dirs, files in os.walk(path):
        for file in files:
            if re.search(r'\d+\.bim$', file):
                bim_files.append(os.path.join(root, file))

data = []

for path in bim_files:
    basename = os.path.basename(path)
    chrom = basename.split('.')[-2]
    data.append({'geno_list': path, 'chr': chrom})

region = pd.DataFrame(data)
region = region.groupby('chr')['geno_list'].apply(lambda x: x.tolist()).reset_index()

regional_data = {
    'geno_list': [row['geno_list'] for _, row in region.iterrows()],
    'chr': [row['chr'] for _, row in region.iterrows()]
}

chr_info = regional_data['chr']

input: regional_data["geno_list"], group_by = lambda x: group_by_region(x, regional_data["geno_list"]), group_with = "chr_info"
# a funtion to get geno prefixs as genos

geno_list = regional_data['geno_list'][0]
genos = '.'.join([os.path.basename(path).split('.')[0] for path in geno_list])
# chrom = os.path.basename(geno_list[0]).split('.')[-2]

output: f"{cwd}/{name}.{genos}.{_chr_info}.aligned.bim.gz", f"{cwd}/{name}.{genos}.{_chr_info}.aligned.bim.gz.tbi"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output[0]:nn}.stdout", stderr = f"{_output[0]:nn}.stderr", container = container
    library(tidyverse) 
    library(data.table)
  
    merge_genotype_data <- function(df1, df2, all = TRUE) {
      setDT(df1)
      setDT(df2)
      df1[, key := paste(chr, pos, pmin(alt, ref), pmax(alt, ref))]
      df2[, key := paste(chr, pos, pmin(alt, ref), pmax(alt, ref))]
      df2[df1, on = "key", flip := i.alt == ref & i.ref == alt, by = .EACHI]
      df2[flip == TRUE, c("alt", "ref") := .(ref, alt)]
      if (all) {
        df_combined <- unique(rbindlist(list(df1[, .(chr, pos, alt, ref)], df2[, .(chr, pos, alt, ref)])), by = c("chr", "pos", "alt", "ref"))
      } else {
        df_combined <- df2[, .(chr, pos, alt, ref)]
      }
      return(df_combined)
    }

   
    bim_files = c(${",".join(['"%s"' % x.absolute() for x in _input])})
    rosmap_bim <- fread(bim_files[1], header = FALSE) 
    colnames(rosmap_bim) <- c('chr', 'id', 'Mics', 'pos', 'alt', 'ref')
    if(length(bim_files) > 1 ){
        for(i in 2:length(bim_files)){
          message('Aligning ', bim_files[i])
          tmp_bim <- fread(bim_files[i], header = FALSE)
          colnames(tmp_bim) <- c('chr', 'id', 'Mics', 'pos', 'alt', 'ref')
          rosmap_bim <- merge_genotype_data(rosmap_bim, tmp_bim)
        }
    } else {
        rosmap_bim <- rosmap_bim[, .(chr, pos, alt, ref)]
    }
    rosmap_bim <- rosmap_bim %>% arrange(pos)
    fwrite(rosmap_bim, ${_output[0]:nr}, sep = '\t', col.names = FALSE)
    system(paste("bgzip ", ${_output[0]:nr}))
    system(paste("tabix -s 1 -b 2 -e 2 ", ${_output[0]:r}))